# RAG Module · Day 24 — **RAG Evaluation Framework (RAGAS)**

**Phase 1 · Module 3: Prompt Engineering & RAG Architecture** · ~2.5 hrs

You've built retrieval (hybrid, graph, agentic) and fit it in the context window. Now the question every Barclays reviewer asks: **how do you know it's correct?** *"Looks good"* isn't an answer. **RAGAS** scores a RAG pipeline on four axes that pinpoint *which stage* fails — retrieval or generation.

Today:
1. The four metrics — **faithfulness, answer relevancy, context precision, context recall** — and which stage each blames.
2. Build a labelled eval set (question, answer, contexts, ground_truth).
3. Score a **faithful** answer vs a **hallucinated** one — watch the metrics separate them.
4. Aggregate across the set into a one-page report.

> We implement the metrics from first principles (token-overlap stand-ins) so it runs keyless. Real RAGAS uses an LLM judge — the *shape* (same four scores, same 0–1 scale) is identical, and the real API is shown in a guarded cell.

## 1. The four metrics — and what each blames

| Metric | Question it asks | Blames | Uses |
|---|---|---|---|
| **Faithfulness** | Is every claim in the answer *supported by the retrieved context*? | generation (hallucination) | answer, contexts |
| **Answer relevancy** | Does the answer actually *address the question*? | generation (evasion) | answer, question |
| **Context precision** | Are the *retrieved* chunks relevant (not noise)? | retrieval (over-fetch) | contexts, ground_truth |
| **Context recall** | Did retrieval fetch *all* the needed info? | retrieval (miss) | contexts, ground_truth |

The power: a low **faithfulness** but high **context recall** means retrieval worked and the *model* hallucinated — you fix the prompt, not the retriever. The scores localise the bug.

In [6]:
import re, statistics

def toks(s): return set(re.findall(r"[a-z0-9]+", s.lower()))
STOP = set("a an the is are was of to and or for in on at it its this that with be as by".split())
def content(s): return toks(s) - STOP

def sentences(s):                                  # split into atomic claims
    return [x.strip() for x in re.split(r"[.;,]| and ", s) if x.strip()]
def numbers(s): return set(re.findall(r"\d+", s))  # figures a claim asserts
print("helpers ready")

helpers ready


☝ `content()` drops stopwords so overlap reflects *facts*, not grammar. `sentences()` splits an answer into claims — faithfulness scores each claim against the context. These stand in for RAGAS's LLM judge, keeping the same 0–1 output.

## 2. The metrics, implemented

Each returns 0–1. Real RAGAS asks an LLM *"is this claim supported?"*; we approximate support by content-token overlap — crude, but it separates a grounded answer from a fabricated one, which is the lesson.

In [7]:
def faithfulness(answer, contexts):
    ctx, ctx_nums = content(" ".join(contexts)), numbers(" ".join(contexts))
    claims = [c for c in sentences(answer) if content(c)]
    if not claims: return 0.0
    # a claim is supported only if its content overlaps AND every figure it cites is in the context
    supported = sum(1 for c in claims
                    if len(content(c) & ctx) / len(content(c)) >= 0.6
                    and numbers(c) <= ctx_nums)
    return supported / len(claims)

def answer_relevancy(answer, question):
    q, a = content(question), content(answer)
    if not q or not a: return 0.0
    return len(q & a) / len(q)                     # fraction of question terms addressed

def context_precision(contexts, ground_truth):
    gt = content(ground_truth)
    if not contexts: return 0.0
    relevant = sum(1 for c in contexts if content(c) & gt)   # chunk touches the truth?
    return relevant / len(contexts)                # 1.0 = no noise chunks

def context_recall(contexts, ground_truth):
    gt = content(ground_truth)
    if not gt: return 0.0
    got = content(" ".join(contexts))
    return len(gt & got) / len(gt)                 # fraction of truth-facts retrieved

print("4 metrics defined")

4 metrics defined


☝ Note which inputs each uses: faithfulness/relevancy judge the **generation**, precision/recall judge the **retrieval** (against `ground_truth`). Same split RAGAS uses. Now watch them react to a good vs bad answer.

## 3. Faithful answer vs hallucinated answer

Same question and retrieved context. Answer A sticks to the context; Answer B invents a figure not in it. Faithfulness should crater on B while relevancy stays high (B *sounds* on-topic) — exactly how you catch a confident hallucination.

In [8]:
question = "What is the overdraft limit on a Premier Current Account?"
contexts = [
    "The Premier Current Account has an arranged overdraft limit of 3000 pounds.",
    "Overdraft interest on the Premier account is charged daily.",
]
ground_truth = "The Premier Current Account overdraft limit is 3000 pounds."

answer_good = "The Premier Current Account has an overdraft limit of 3000 pounds, with interest charged daily."
answer_hallucinated = "The Premier Current Account overdraft limit is 8000 pounds and comes with free travel insurance."

for label, ans in [("faithful", answer_good), ("hallucinated", answer_hallucinated)]:
    print(f"--- {label} ---")
    print(f"  faithfulness    : {faithfulness(ans, contexts):.2f}")
    print(f"  answer_relevancy: {answer_relevancy(ans, question):.2f}")
    print(f"  context_precision:{context_precision(contexts, ground_truth):.2f}")
    print(f"  context_recall  : {context_recall(contexts, ground_truth):.2f}")

--- faithful ---
  faithfulness    : 1.00
  answer_relevancy: 0.83
  context_precision:1.00
  context_recall  : 1.00
--- hallucinated ---
  faithfulness    : 0.00
  answer_relevancy: 0.83
  context_precision:1.00
  context_recall  : 1.00


☝ Both answers score high on **relevancy** (both mention overdraft/Premier) and share identical **retrieval** scores (same contexts) — but **faithfulness** collapses on the hallucinated answer because *"8000 pounds"* and *"free travel insurance"* aren't in the context. That one metric is your hallucination alarm; the flat retrieval scores prove the retriever wasn't at fault.

## 4. Aggregate across an eval set → the report

RAGAS runs over a **dataset** of examples and reports the mean per metric — the one-page number you take to a review. We score a small labelled banking set; in production this is 20+ human-labelled pairs against the Day-18 Financial RAG Agent.

In [9]:
# labelled eval set: each row = (question, contexts, answer, ground_truth)
DATASET = [
    ("ISA annual allowance?",
     ["The annual ISA allowance is 20000 pounds per tax year."],
     "The annual ISA allowance is 20000 pounds.",
     "The ISA allowance is 20000 pounds per year."),
    ("Minimum residential mortgage deposit?",
     ["Residential mortgages require a minimum deposit of five percent."],
     "You need a minimum deposit of five percent.",
     "The minimum deposit is five percent."),
    ("Contactless payment cap?",
     ["Contactless payments are capped at 100 pounds per transaction.",
      "Report a lost card immediately."],                     # 2nd chunk = noise
     "Contactless is capped at 100 pounds.",
     "The contactless limit is 100 pounds."),
    ("Buy-to-let deposit?",
     ["Buy-to-let mortgages require a 25 percent deposit."],
     "Buy-to-let needs a 40 percent deposit.",                # WRONG figure -> hallucination
     "Buy-to-let requires a 25 percent deposit."),
]

metrics = {"faithfulness": [], "answer_relevancy": [], "context_precision": [], "context_recall": []}
print(f"{'question':32}{'faith':>7}{'relev':>7}{'prec':>7}{'recall':>7}")
for q, ctx, ans, gt in DATASET:
    row = {
        "faithfulness": faithfulness(ans, ctx),
        "answer_relevancy": answer_relevancy(ans, q),
        "context_precision": context_precision(ctx, gt),
        "context_recall": context_recall(ctx, gt),
    }
    for k, v in row.items(): metrics[k].append(v)
    print(f"{q[:32]:32}{row['faithfulness']:7.2f}{row['answer_relevancy']:7.2f}"
          f"{row['context_precision']:7.2f}{row['context_recall']:7.2f}")

print("\n=== RAG EVAL REPORT (mean) ===")
for k, vals in metrics.items():
    print(f"  {k:18}: {statistics.mean(vals):.2f}")

question                          faith  relev   prec recall
ISA annual allowance?              1.00   1.00   1.00   1.00
Minimum residential mortgage dep   1.00   0.50   1.00   1.00
Contactless payment cap?           1.00   0.33   0.50   0.75
Buy-to-let deposit?                0.00   1.00   1.00   0.83

=== RAG EVAL REPORT (mean) ===
  faithfulness      : 0.75
  answer_relevancy  : 0.71
  context_precision : 0.88
  context_recall    : 0.90


☝ The report surfaces two defects at a glance: row 4's **faithfulness** drops (the *40 percent* hallucination), and row 3's **context_precision** drops (the lost-card noise chunk). Low faithfulness → fix generation; low precision → tighten retrieval/reranking. The mean row is your pipeline's scorecard, tracked release over release.

## 5. The real RAGAS API (for reference)

In production you don't hand-roll the metrics — RAGAS wraps them around an LLM judge and your embedder. The dataset shape is exactly what we built above.

In [10]:
ragas_snippet = '''
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset

ds = Dataset.from_dict({
    "question":     [q for q, _, _, _ in DATASET],
    "contexts":     [c for _, c, _, _ in DATASET],
    "answer":       [a for _, _, a, _ in DATASET],
    "ground_truth": [g for _, _, _, g in DATASET],
})
result = evaluate(ds, metrics=[faithfulness, answer_relevancy,
                               context_precision, context_recall])
print(result)   # {'faithfulness': 0.85, 'answer_relevancy': 0.91, ...}
'''
import importlib.util
if importlib.util.find_spec("ragas") is None:
    print("ragas NOT installed -> pip install ragas datasets")
else:
    try:
        import ragas
        print(f"ragas {ragas.__version__} installed and importable "
              "— the snippet above runs against your judge LLM (needs API keys).")
    except Exception as e:
        # installed, but a transitive import failed (common: ragas pinned to an
        # older langchain path that your langchain-community version has moved/removed)
        print(f"ragas IS installed but failed to import: {type(e).__name__}: {e}")
        print("-> version conflict, not a missing package. Align ragas with your "
              "langchain-community version, or run RAGAS in an isolated env.")
print("\nReference usage (real judge-LLM path):")
print(ragas_snippet)

ragas IS installed but failed to import: ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'
-> version conflict, not a missing package. Align ragas with your langchain-community version, or run RAGAS in an isolated env.

Reference usage (real judge-LLM path):

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset

ds = Dataset.from_dict({
    "question":     [q for q, _, _, _ in DATASET],
    "contexts":     [c for _, c, _, _ in DATASET],
    "answer":       [a for _, _, a, _ in DATASET],
    "ground_truth": [g for _, _, _, g in DATASET],
})
result = evaluate(ds, metrics=[faithfulness, answer_relevancy,
                               context_precision, context_recall])
print(result)   # {'faithfulness': 0.85, 'answer_relevancy': 0.91, ...}



☝ Same four metrics, same dataset columns — RAGAS just swaps our token-overlap for an LLM that reads each claim. On Bedrock you point the judge at Claude and evaluate the Day-18 agent. Whether the judge is a heuristic or an LLM, **the discipline is the same: label a set, score four axes, track the mean.**

## 6. Recap — you can't improve what you don't measure

| Metric | Low score means | Fix |
|---|---|---|
| **Faithfulness** | model hallucinated beyond context | prompt / grounding / guardrails |
| **Answer relevancy** | answer dodged the question | prompt / query understanding |
| **Context precision** | retrieved noise | reranking, higher threshold |
| **Context recall** | missed needed chunks | chunking, hybrid search, higher k |

This closes **Module 3**: retrieve well (Days 20–22), fit the window (Day 23), and **prove it with numbers** (today). An eval score is what turns a RAG demo into a system you can defend in a Barclays review.